# 02 – Data preparation

Reads the typed views in the `analytics` schema (`sql/03_create_typed_views.sql`), checks them against the Stage 1 control totals, adds the new variables (promised and actual delivery days, seller-to-customer distance) and exports two tables to `data/processed/`:

- `orders.csv`: one row per order (99,441 rows, no scope filter).
- `order_items.csv`: one row per order item.

Treatment decisions are logged in `docs/decisions.md`.

## Setup

Imports and database connection. Credentials are read from `.env`, which is not versioned.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

## 1. Read the typed views

The four views of the `analytics` schema are read with pandas nullable dtypes, so integer and boolean columns with missing values keep their type (`Int64`, `boolean`) instead of becoming floats or objects. Expected shapes: `orders` 99,441 × 20, `order_items` 112,650 × 20, `order_reviews_latest` 98,673 × 7, `zip_coordinates` 19,015 × 5.

In [2]:
orders = pd.read_sql_table("orders", engine, schema="analytics", dtype_backend="numpy_nullable")
order_items = pd.read_sql_table("order_items", engine, schema="analytics", dtype_backend="numpy_nullable")
reviews = pd.read_sql_table("order_reviews_latest", engine, schema="analytics", dtype_backend="numpy_nullable")
zip_coords = pd.read_sql_table("zip_coordinates", engine, schema="analytics", dtype_backend="numpy_nullable")

print(f"orders:      {orders.shape}")
print(f"order_items: {order_items.shape}")
print(f"reviews:     {reviews.shape}")
print(f"zip_coords:  {zip_coords.shape}")
print()
print(orders.dtypes.to_string())
print()
print(zip_coords.dtypes.to_string())

orders:      (99441, 20)
order_items: (112650, 20)
reviews:     (98673, 7)
zip_coords:  (19015, 5)

order_id                                 string
customer_id                              string
customer_unique_id                       string
customer_zip_code_prefix                 string
customer_city                            string
customer_state                           string
order_status                             string
purchase_ts                      datetime64[us]
approved_ts                      datetime64[us]
carrier_ts                       datetime64[us]
delivered_ts                     datetime64[us]
estimated_date                   datetime64[us]
purchase_month                   datetime64[us]
in_delay_scope                          boolean
flag_approval_before_purchase           boolean
flag_carrier_before_approval            boolean
flag_carrier_before_purchase            boolean
flag_delivery_before_carrier            boolean
is_late                             

## 2. Control totals at order grain

One row per order. Each count is compared across three independent sources: the reference value in `docs/decisions.md`, a control SQL query run now, and pandas. The late flag follows the date rule (D9) and the delay scope follows D2. Every check ends in an `assert`, so the notebook stops if a total does not match.

In [3]:
EXPECTED = {
    "orders": 99441,
    "delivered": 96478,
    "delay_scope": 96470,
    "late": 6534,
    "delivered_on_estimated_date": 1292,
    "inconsistent_timestamps": 1373,
}

control_sql = """
SELECT
    COUNT(*) AS orders,
    COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered,
    COUNT(*) FILTER (WHERE in_delay_scope) AS delay_scope,
    COUNT(*) FILTER (WHERE in_delay_scope AND is_late) AS late,
    COUNT(*) FILTER (WHERE in_delay_scope AND delivery_delay_days = 0) AS delivered_on_estimated_date,
    COUNT(*) FILTER (
        WHERE order_status = 'delivered'
          AND (flag_approval_before_purchase OR flag_carrier_before_approval
               OR flag_carrier_before_purchase OR flag_delivery_before_carrier)
    ) AS inconsistent_timestamps
FROM analytics.orders;
"""
observed_sql = pd.read_sql(text(control_sql), engine).iloc[0].to_dict()

flag_cols = [
    "flag_approval_before_purchase",
    "flag_carrier_before_approval",
    "flag_carrier_before_purchase",
    "flag_delivery_before_carrier",
]
is_delivered = orders["order_status"] == "delivered"
in_scope = orders["in_delay_scope"].fillna(False)
is_late = in_scope & orders["is_late"].fillna(False)
on_estimated_date = in_scope & (orders["delivery_delay_days"] == 0).fillna(False)
inconsistent = is_delivered & orders[flag_cols].fillna(False).any(axis=1)

observed_pandas = {
    "orders": len(orders),
    "delivered": int(is_delivered.sum()),
    "delay_scope": int(in_scope.sum()),
    "late": int(is_late.sum()),
    "delivered_on_estimated_date": int(on_estimated_date.sum()),
    "inconsistent_timestamps": int(inconsistent.sum()),
}

results = pd.DataFrame({"expected": EXPECTED, "sql": observed_sql, "pandas": observed_pandas})
matches = (results["expected"] == results["sql"]) & (results["expected"] == results["pandas"])
results["status"] = matches.map({True: "PASS", False: "FAIL"})
print(results.to_string())

late_rate = round(is_late.sum() / in_scope.sum() * 100, 2)
print(f"\nLate rate: {late_rate}% (expected 6.77%)")

# D9 consistency inside the scope: late <=> positive delay
scope_rows = orders[in_scope]
d9_consistent = (scope_rows["is_late"] == (scope_rows["delivery_delay_days"] > 0)).all()
print(f"order_id unique: {orders['order_id'].is_unique}")
print(f"is_late consistent with delivery_delay_days > 0: {d9_consistent}")

assert matches.all(), "Control totals do not match"
assert late_rate == 6.77
assert orders["order_id"].is_unique
assert d9_consistent

                             expected    sql  pandas status
orders                          99441  99441   99441   PASS
delivered                       96478  96478   96478   PASS
delay_scope                     96470  96470   96470   PASS
late                             6534   6534    6534   PASS
delivered_on_estimated_date      1292   1292    1292   PASS
inconsistent_timestamps          1373   1373    1373   PASS

Late rate: 6.77% (expected 6.77%)
order_id unique: True
is_late consistent with delivery_delay_days > 0: True


## 3. Item grain: copied order columns and values

One row per item. Order-level columns (`order_status`, `in_delay_scope`, `is_late`, `delivery_delay_days`) are copied from `orders` with a many-to-one merge, never recalculated; the merge must not change the row count. Price and freight are reconciled within R$ 0.01, and the freight share is a ratio of sums (D4).

In [4]:
TOLERANCE_BRL = 0.01

order_level_cols = ["order_id", "order_status", "in_delay_scope", "is_late", "delivery_delay_days"]
rows_before = len(order_items)
items = order_items.merge(orders[order_level_cols], on="order_id", how="left", validate="many_to_one")

print(f"Rows before merge: {rows_before} | after merge: {len(items)}")
print(f"Item key (order_id, order_item_id) unique: {not items.duplicated(['order_id', 'order_item_id']).any()}")
print(f"Items without a matching order: {items['order_status'].isna().sum()}")

control_sql_items = """
SELECT
    COUNT(*) AS item_rows,
    COUNT(*) FILTER (WHERE o.in_delay_scope) AS items_in_scope,
    COUNT(*) FILTER (WHERE o.order_status = 'delivered' AND i.category = 'unknown') AS unknown_item_rows_delivered,
    COUNT(DISTINCT i.order_id) FILTER (WHERE o.order_status = 'delivered' AND i.category = 'unknown') AS unknown_orders_delivered,
    SUM(i.price) FILTER (WHERE o.in_delay_scope) AS price_in_scope,
    SUM(i.freight_value) FILTER (WHERE o.in_delay_scope) AS freight_in_scope
FROM analytics.order_items AS i
JOIN analytics.orders AS o ON o.order_id = i.order_id;
"""
sql_items = pd.read_sql(text(control_sql_items), engine).iloc[0].to_dict()

item_in_scope = items["in_delay_scope"].fillna(False)
item_delivered = items["order_status"] == "delivered"
item_unknown = item_delivered & (items["category"] == "unknown")

# Counts: exact match
EXPECTED_ITEM_COUNTS = {
    "item_rows": 112650,
    "items_in_scope": 110189,
    "unknown_item_rows_delivered": 1537,
    "unknown_orders_delivered": 1392,
}
pandas_item_counts = {
    "item_rows": len(items),
    "items_in_scope": int(item_in_scope.sum()),
    "unknown_item_rows_delivered": int(item_unknown.sum()),
    "unknown_orders_delivered": items.loc[item_unknown, "order_id"].nunique(),
}
counts = pd.DataFrame({
    "expected": EXPECTED_ITEM_COUNTS,
    "sql": {k: int(sql_items[k]) for k in EXPECTED_ITEM_COUNTS},
    "pandas": pandas_item_counts,
})
counts_ok = (counts["expected"] == counts["sql"]) & (counts["expected"] == counts["pandas"])
counts["status"] = counts_ok.map({True: "PASS", False: "FAIL"})
print("\n" + counts.to_string())

# Values: match within TOLERANCE_BRL
EXPECTED_VALUES = {"price_in_scope": 13220248.93, "freight_in_scope": 2198145.90}
pandas_values = {
    "price_in_scope": items.loc[item_in_scope, "price"].sum(),
    "freight_in_scope": items.loc[item_in_scope, "freight_value"].sum(),
}
values = pd.DataFrame({
    "expected": EXPECTED_VALUES,
    "sql": {k: float(sql_items[k]) for k in EXPECTED_VALUES},
    "pandas": {k: float(v) for k, v in pandas_values.items()},
})
values_ok = ((values["sql"] - values["expected"]).abs() <= TOLERANCE_BRL) & (
    (values["pandas"] - values["expected"]).abs() <= TOLERANCE_BRL
)
values["status"] = values_ok.map({True: "PASS", False: "FAIL"})
print("\n" + values.round(2).to_string())

freight_share = round(
    pandas_values["freight_in_scope"] / (pandas_values["price_in_scope"] + pandas_values["freight_in_scope"]) * 100, 2
)
print(f"\nFreight share: {freight_share}% (expected 14.26%)")

assert len(items) == rows_before
assert not items.duplicated(["order_id", "order_item_id"]).any()
assert items["order_status"].notna().all()
assert counts_ok.all() and values_ok.all()
assert freight_share == 14.26

Rows before merge: 112650 | after merge: 112650
Item key (order_id, order_item_id) unique: True
Items without a matching order: 0

                             expected     sql  pandas status
item_rows                      112650  112650  112650   PASS
items_in_scope                 110189  110189  110189   PASS
unknown_item_rows_delivered      1537    1537    1537   PASS
unknown_orders_delivered         1392    1392    1392   PASS

                     expected          sql       pandas status
price_in_scope    13220248.93  13220248.93  13220248.93   PASS
freight_in_scope   2198145.90   2198145.90   2198145.90   PASS

Freight share: 14.26% (expected 14.26%)


## 4. Reviews

The latest review per order (D5) is added to the order table with a one-to-one merge, which also confirms that `order_reviews_latest` has one row per order. Average scores compare late and not-late orders inside the delay scope.

In [5]:
orders_prep = orders.merge(
    reviews[["order_id", "review_score"]], on="order_id", how="left", validate="one_to_one"
)
print(f"Rows before merge: {len(orders)} | after merge: {len(orders_prep)}")

control_sql_reviews = """
SELECT
    COUNT(*) FILTER (WHERE r.order_id IS NULL) AS orders_without_review,
    COUNT(*) FILTER (WHERE o.in_delay_scope AND r.order_id IS NOT NULL) AS reviewed_in_scope,
    COUNT(*) FILTER (WHERE o.in_delay_scope AND r.order_id IS NOT NULL AND o.is_late) AS reviewed_late,
    COUNT(*) FILTER (WHERE o.in_delay_scope AND r.order_id IS NOT NULL AND NOT o.is_late) AS reviewed_not_late,
    ROUND(AVG(r.review_score) FILTER (WHERE o.in_delay_scope AND o.is_late), 2) AS avg_score_late,
    ROUND(AVG(r.review_score) FILTER (WHERE o.in_delay_scope AND NOT o.is_late), 2) AS avg_score_not_late
FROM analytics.orders AS o
LEFT JOIN analytics.order_reviews_latest AS r ON r.order_id = o.order_id;
"""
sql_reviews = pd.read_sql(text(control_sql_reviews), engine).iloc[0].to_dict()

has_review = orders_prep["review_score"].notna()
prep_in_scope = orders_prep["in_delay_scope"].fillna(False)
prep_late = prep_in_scope & orders_prep["is_late"].fillna(False)
prep_not_late = prep_in_scope & ~orders_prep["is_late"].fillna(True)

EXPECTED_REVIEWS = {
    "orders_without_review": 768,
    "reviewed_in_scope": 95824,
    "reviewed_late": 6381,
    "reviewed_not_late": 89443,
    "avg_score_late": 2.27,
    "avg_score_not_late": 4.29,
}
pandas_reviews = {
    "orders_without_review": int((~has_review).sum()),
    "reviewed_in_scope": int((prep_in_scope & has_review).sum()),
    "reviewed_late": int((prep_late & has_review).sum()),
    "reviewed_not_late": int((prep_not_late & has_review).sum()),
    "avg_score_late": round(float(orders_prep.loc[prep_late, "review_score"].mean()), 2),
    "avg_score_not_late": round(float(orders_prep.loc[prep_not_late, "review_score"].mean()), 2),
}
review_check = pd.DataFrame({
    "expected": EXPECTED_REVIEWS,
    "sql": {k: float(v) for k, v in sql_reviews.items()},
    "pandas": pandas_reviews,
})
review_ok = (review_check["expected"] == review_check["sql"]) & (review_check["expected"] == review_check["pandas"])
review_check["status"] = review_ok.map({True: "PASS", False: "FAIL"})
print("\n" + review_check.to_string())
print(f"\nreview_score dtype: {orders_prep['review_score'].dtype}")

assert len(orders_prep) == len(orders)
assert review_ok.all()

Rows before merge: 99441 | after merge: 99441

                       expected       sql    pandas status
orders_without_review    768.00    768.00    768.00   PASS
reviewed_in_scope      95824.00  95824.00  95824.00   PASS
reviewed_late           6381.00   6381.00   6381.00   PASS
reviewed_not_late      89443.00  89443.00  89443.00   PASS
avg_score_late             2.27      2.27      2.27   PASS
avg_score_not_late         4.29      4.29      4.29   PASS

review_score dtype: Int64


## 5. Promised and actual delivery days (D13)

Rule set before computing: both variables are whole days by date, consistent with D9. `promised_delivery_days` (estimated date minus purchase date) exists for every order; `actual_delivery_days` (delivery date minus purchase date) only for orders in the delay scope. Check: actual minus promised must equal `delivery_delay_days` for every order in scope.

In [6]:
purchase_date = orders_prep["purchase_ts"].dt.normalize()
delivered_date = orders_prep["delivered_ts"].dt.normalize()

orders_prep["promised_delivery_days"] = (orders_prep["estimated_date"] - purchase_date).dt.days.astype("Int64")
orders_prep["actual_delivery_days"] = (
    (delivered_date - purchase_date).dt.days.astype("Int64").where(prep_in_scope)
)

scope_prep = orders_prep[prep_in_scope]
identity_ok = (
    scope_prep["actual_delivery_days"] - scope_prep["promised_delivery_days"] == scope_prep["delivery_delay_days"]
).all()

print(f"Missing promised_delivery_days (all orders): {orders_prep['promised_delivery_days'].isna().sum()}")
print(f"Missing actual_delivery_days (all orders):   {orders_prep['actual_delivery_days'].isna().sum()}")
print(f"Missing actual_delivery_days (delay scope):  {scope_prep['actual_delivery_days'].isna().sum()}")
print(f"Identity holds for all {len(scope_prep)} orders in scope: {identity_ok}")
print()
print(scope_prep[["promised_delivery_days", "actual_delivery_days", "delivery_delay_days"]].describe().round(2).to_string())

assert identity_ok
assert scope_prep["actual_delivery_days"].notna().all()

Missing promised_delivery_days (all orders): 0
Missing actual_delivery_days (all orders):   2971
Missing actual_delivery_days (delay scope):  0
Identity holds for all 96470 orders in scope: True

       promised_delivery_days  actual_delivery_days  delivery_delay_days
count                 96470.0               96470.0              96470.0
mean                    24.37                  12.5               -11.88
std                      8.76                  9.56                10.18
min                       3.0                   0.0               -147.0
25%                      19.0                   7.0                -17.0
50%                      24.0                  10.0                -12.0
75%                      29.0                  16.0                 -7.0
max                     156.0                 210.0                188.0


## 6. Seller-to-customer distance (D14)

Rules set before measuring: (1) at item grain, straight-line (haversine) distance between the seller and customer ZIP coordinates; (2) at order grain, the farthest item, because a multi-seller order is complete only when the last package arrives; (3) missing if any item of the order has no distance. Orders without distance are measured by two independent paths, the calculation and the ZIP groups, which must give the same set.

In [7]:
zip_col = zip_coords.columns[0]
print(f"zip_coordinates columns: {zip_coords.columns.tolist()}")
assert zip_col.endswith("zip_code_prefix"), "Unexpected first column in zip_coordinates"

coords = zip_coords[[zip_col, "latitude", "longitude"]]
seller_coords = coords.rename(columns={zip_col: "seller_zip_code_prefix", "latitude": "seller_lat", "longitude": "seller_lng"})
customer_coords = coords.rename(columns={zip_col: "customer_zip_code_prefix", "latitude": "customer_lat", "longitude": "customer_lng"})

rows_before = len(items)
items_geo = (
    items
    .merge(orders[["order_id", "customer_zip_code_prefix"]], on="order_id", how="left", validate="many_to_one")
    .merge(seller_coords, on="seller_zip_code_prefix", how="left", validate="many_to_one")
    .merge(customer_coords, on="customer_zip_code_prefix", how="left", validate="many_to_one")
)
assert len(items_geo) == rows_before

EARTH_RADIUS_KM = 6371.0

def haversine_km(lat1, lng1, lat2, lng2):
    """Straight-line (great-circle) distance in km between two points in decimal degrees."""
    lat1, lng1, lat2, lng2 = np.radians(lat1), np.radians(lng1), np.radians(lat2), np.radians(lng2)
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

items_geo["distance_km"] = haversine_km(
    items_geo["seller_lat"], items_geo["seller_lng"], items_geo["customer_lat"], items_geo["customer_lng"]
)

order_distance = items_geo.groupby("order_id").agg(
    max_item_distance_km=("distance_km", "max"),
    n_items=("distance_km", "size"),
    n_items_with_distance=("distance_km", "count"),
).reset_index()
order_distance["distance_km"] = order_distance["max_item_distance_km"].where(
    order_distance["n_items_with_distance"] == order_distance["n_items"]
)

orders_prep = orders_prep.drop(columns=["distance_km"], errors="ignore").merge(
    order_distance[["order_id", "distance_km"]], on="order_id", how="left", validate="one_to_one"
)
assert len(orders_prep) == 99441

# Measurement in the delay scope: two independent paths to the orders without distance
scope_orders = orders_prep[orders_prep["in_delay_scope"].fillna(False)]
scope_items = items_geo[items_geo["in_delay_scope"].fillna(False)]
known_zips = zip_coords[zip_col]
null_zips = zip_coords.loc[zip_coords["latitude"].isna(), zip_col]

customer_no_match = set(scope_orders.loc[~scope_orders["customer_zip_code_prefix"].isin(known_zips), "order_id"])
seller_no_match = set(scope_items.loc[~scope_items["seller_zip_code_prefix"].isin(known_zips), "order_id"])
null_zip_orders = (
    set(scope_orders.loc[scope_orders["customer_zip_code_prefix"].isin(null_zips), "order_id"])
    | set(scope_items.loc[scope_items["seller_zip_code_prefix"].isin(null_zips), "order_id"])
)
missing_by_zip_groups = customer_no_match | seller_no_match | null_zip_orders
missing_by_calculation = set(scope_orders.loc[scope_orders["distance_km"].isna(), "order_id"])
multi_seller_orders = int(scope_items.groupby("order_id")["seller_id"].nunique().gt(1).sum())

print(f"\nCustomer ZIP without match:              {len(customer_no_match)} (audit: 264)")
print(f"Seller ZIP without match:                {len(seller_no_match)} (audit: 217)")
print(f"Overlap of the two groups:               {len(customer_no_match & seller_no_match)}")
print(f"Orders touching the 5 ZIPs with no valid point: {len(null_zip_orders)}")
print(f"  of which outside the two groups above: {len(null_zip_orders - customer_no_match - seller_no_match)}")
print(f"Orders without distance (calculation):   {len(missing_by_calculation)} "
      f"({len(missing_by_calculation) / len(scope_orders) * 100:.2f}% of {len(scope_orders)})")
print(f"Same set by both paths:                  {missing_by_calculation == missing_by_zip_groups}")
print(f"Orders without distance equal to 481 (D14): {len(missing_by_calculation) == 481}")
print(f"Orders with more than one seller:        {multi_seller_orders} (expected 1,275)")
print()
print(scope_orders["distance_km"].describe().round(1).to_string())

assert missing_by_calculation == missing_by_zip_groups
assert multi_seller_orders == 1275
assert scope_orders["distance_km"].max() < 5000
assert len(missing_by_calculation) == 481

zip_coordinates columns: ['zip_code_prefix', 'latitude', 'longitude', 'point_count', 'valid_point_count']

Customer ZIP without match:              264 (audit: 264)
Seller ZIP without match:                217 (audit: 217)
Overlap of the two groups:               1
Orders touching the 5 ZIPs with no valid point: 1
  of which outside the two groups above: 1
Orders without distance (calculation):   481 (0.50% of 96470)
Same set by both paths:                  True
Orders without distance equal to 481 (D14): True
Orders with more than one seller:        1275 (expected 1,275)

count    95989.0
mean       601.8
std        592.5
min          0.0
25%        189.4
50%        435.6
75%        800.2
max       3399.2


## 7. Export (D15)

Two tables are written to `data/processed/`: `orders.csv` (order grain, no scope filter) and `order_items.csv` (item grain). Money columns exist only in `order_items` (D4); `in_delay_scope`, `is_late`, `delivery_delay_days` and `customer_state` are copied into it from `orders`. Files are UTF-8 with BOM for Power Query. They are not versioned: they are regenerated by the load notebook (`01_load_staging.ipynb`), the SQL scripts that create the staging tables and the typed views (`sql/01`, `sql/03`) and this notebook.

In [8]:
ORDERS_COLUMNS = [
    "order_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state",
    "order_status", "purchase_ts", "approved_ts", "carrier_ts", "delivered_ts", "estimated_date", "purchase_month",
    "in_delay_scope", "flag_approval_before_purchase", "flag_carrier_before_approval",
    "flag_carrier_before_purchase", "flag_delivery_before_carrier",
    "is_late", "delivery_delay_days", "promised_delivery_days", "actual_delivery_days",
    "distance_km", "has_distance", "review_score",
]
ITEMS_COLUMNS = [
    "order_id", "order_item_id", "product_id", "seller_id", "seller_zip_code_prefix", "seller_city", "seller_state",
    "shipping_limit_ts", "price", "freight_value", "category", "category_source",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    "customer_state", "in_delay_scope", "is_late", "delivery_delay_days", "distance_km",
]

orders_export = orders_prep.assign(
    has_distance=orders_prep["distance_km"].notna(),
    distance_km=orders_prep["distance_km"].round(2),
)[ORDERS_COLUMNS]

# Item grain: customer_state is copied from orders, never recalculated
items_export = items_geo.merge(
    orders[["order_id", "customer_state"]], on="order_id", how="left", validate="many_to_one"
).assign(distance_km=lambda df: df["distance_km"].round(2))[ITEMS_COLUMNS]

assert len(orders_export) == 99441 and orders_export["order_id"].is_unique
assert len(items_export) == 112650 and not items_export.duplicated(["order_id", "order_item_id"]).any()

PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

orders_path = PROCESSED_DIR / "orders.csv"
items_path = PROCESSED_DIR / "order_items.csv"
orders_export.to_csv(orders_path, index=False, encoding="utf-8-sig")
items_export.to_csv(items_path, index=False, encoding="utf-8-sig")

for path in [orders_path, items_path]:
    size_mb = path.stat().st_size / 1024**2
    print(f"{path.name}: {size_mb:.1f} MB")

orders.csv: 24.3 MB
order_items.csv: 24.0 MB


## 8. Exported file format

The files are re-read as raw text, with no type conversion, to check what Excel and Power BI will receive: BOM, five-digit ZIP prefixes with leading zeros, integers without `.0`, ISO dates and `True`/`False` booleans.

In [9]:
raw_orders = pd.read_csv(orders_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
raw_items = pd.read_csv(items_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

def has_bom(path):
    with open(path, "rb") as f:
        return f.read(3) == b"\xef\xbb\xbf"

def all_match(series, pattern, allow_empty=False):
    """True if every value matches the regular expression (optionally allowing empty strings)."""
    values = series[series != ""] if allow_empty else series
    return bool(values.str.fullmatch(pattern).all())

INTEGER = r"-?\d+"
DATE = r"\d{4}-\d{2}-\d{2}"
TIMESTAMP = r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
ZIP = r"\d{5}"

checks = {
    "orders: rows": len(raw_orders) == 99441,
    "items: rows": len(raw_items) == 112650,
    "orders: UTF-8 BOM": has_bom(orders_path),
    "items: UTF-8 BOM": has_bom(items_path),
    "orders: customer ZIP is 5 digits": all_match(raw_orders["customer_zip_code_prefix"], ZIP),
    "items: seller ZIP is 5 digits": all_match(raw_items["seller_zip_code_prefix"], ZIP),
    "orders: estimated_date and purchase_month as YYYY-MM-DD": (
        all_match(raw_orders["estimated_date"], DATE) and all_match(raw_orders["purchase_month"], DATE)
    ),
    "orders: timestamps as YYYY-MM-DD HH:MM:SS": all(
        all_match(raw_orders[c], TIMESTAMP, allow_empty=True)
        for c in ["purchase_ts", "approved_ts", "carrier_ts", "delivered_ts"]
    ),
    "orders: integer columns without .0": all(
        all_match(raw_orders[c], INTEGER, allow_empty=True)
        for c in ["delivery_delay_days", "promised_delivery_days", "actual_delivery_days", "review_score"]
    ),
    "items: integer columns without .0": all(
        all_match(raw_items[c], INTEGER, allow_empty=True) for c in ["order_item_id", "delivery_delay_days"]
    ),
    "orders: booleans are True/False/empty": all(
        set(raw_orders[c].unique()) <= {"True", "False", ""}
        for c in ["in_delay_scope", "is_late", "has_distance"] + flag_cols
    ),
    "orders: in_delay_scope and has_distance never empty": (
        (raw_orders["in_delay_scope"] != "").all() and (raw_orders["has_distance"] != "").all()
    ),
}

check_table = pd.Series(checks).map({True: "PASS", False: "FAIL"})
print(check_table.to_string())
print()
print(raw_orders[["customer_zip_code_prefix", "estimated_date", "purchase_ts", "is_late",
                  "delivery_delay_days", "review_score", "distance_km"]].head(3).to_string(index=False))

assert all(checks.values()), "Exported file format check failed"

orders: rows                                               PASS
items: rows                                                PASS
orders: UTF-8 BOM                                          PASS
items: UTF-8 BOM                                           PASS
orders: customer ZIP is 5 digits                           PASS
items: seller ZIP is 5 digits                              PASS
orders: estimated_date and purchase_month as YYYY-MM-DD    PASS
orders: timestamps as YYYY-MM-DD HH:MM:SS                  PASS
orders: integer columns without .0                         PASS
items: integer columns without .0                          PASS
orders: booleans are True/False/empty                      PASS
orders: in_delay_scope and has_distance never empty        PASS

customer_zip_code_prefix estimated_date         purchase_ts is_late delivery_delay_days review_score distance_km
                   03149     2017-10-18 2017-10-02 10:56:33   False                  -8            4       18.58
     

## 9. Final reconciliation from the exported files

The files are re-read with explicit column types, as Power Query will read them, and every control total is reconciled again from disk. The average days late is computed per order from both files; the item-weighted average is printed only to show why counting at the wrong grain does not reconcile.

In [10]:
orders_csv = pd.read_csv(
    orders_path, encoding="utf-8-sig",
    dtype={
        "order_id": "string", "customer_unique_id": "string", "customer_zip_code_prefix": "string",
        "customer_city": "string", "customer_state": "string", "order_status": "string",
        "in_delay_scope": "boolean", "is_late": "boolean", "has_distance": "boolean",
        "flag_approval_before_purchase": "boolean", "flag_carrier_before_approval": "boolean",
        "flag_carrier_before_purchase": "boolean", "flag_delivery_before_carrier": "boolean",
        "delivery_delay_days": "Int64", "promised_delivery_days": "Int64",
        "actual_delivery_days": "Int64", "review_score": "Int64", "distance_km": "Float64",
    },
    parse_dates=["purchase_ts", "approved_ts", "carrier_ts", "delivered_ts", "estimated_date", "purchase_month"],
)
items_csv = pd.read_csv(
    items_path, encoding="utf-8-sig",
    dtype={
        "order_id": "string", "product_id": "string", "seller_id": "string",
        "seller_zip_code_prefix": "string", "seller_city": "string", "seller_state": "string",
        "category": "string", "category_source": "string", "customer_state": "string",
        "order_item_id": "Int64", "in_delay_scope": "boolean", "is_late": "boolean",
        "delivery_delay_days": "Int64", "price": "Float64", "freight_value": "Float64", "distance_km": "Float64",
    },
    parse_dates=["shipping_limit_ts"],
)

# Masks
o_delivered = orders_csv["order_status"] == "delivered"
o_scope = orders_csv["in_delay_scope"]
o_late = o_scope & orders_csv["is_late"].fillna(False)
o_not_late = o_scope & ~orders_csv["is_late"].fillna(True)
o_reviewed = orders_csv["review_score"].notna()
i_scope = items_csv["in_delay_scope"].fillna(False)
i_delivered = items_csv["order_id"].isin(orders_csv.loc[o_delivered, "order_id"])
i_unknown = i_delivered & (items_csv["category"] == "unknown")

# 1) Counts: expected values reused from sections 2 to 4
expected_counts = {}
expected_counts.update(EXPECTED)
expected_counts.update(EXPECTED_ITEM_COUNTS)
expected_counts.update({k: EXPECTED_REVIEWS[k] for k in
                        ["orders_without_review", "reviewed_in_scope", "reviewed_late", "reviewed_not_late"]})
expected_counts["missing_distance_in_scope"] = 481

csv_counts = {
    "orders": len(orders_csv),
    "delivered": o_delivered.sum(),
    "delay_scope": o_scope.sum(),
    "late": o_late.sum(),
    "delivered_on_estimated_date": (o_scope & (orders_csv["delivery_delay_days"] == 0).fillna(False)).sum(),
    "inconsistent_timestamps": (o_delivered & orders_csv[flag_cols].fillna(False).any(axis=1)).sum(),
    "item_rows": len(items_csv),
    "items_in_scope": i_scope.sum(),
    "unknown_item_rows_delivered": i_unknown.sum(),
    "unknown_orders_delivered": items_csv.loc[i_unknown, "order_id"].nunique(),
    "orders_without_review": (~o_reviewed).sum(),
    "reviewed_in_scope": (o_scope & o_reviewed).sum(),
    "reviewed_late": (o_late & o_reviewed).sum(),
    "reviewed_not_late": (o_not_late & o_reviewed).sum(),
    "missing_distance_in_scope": (o_scope & ~orders_csv["has_distance"]).sum(),
}
count_table = pd.DataFrame({"expected": expected_counts, "csv": {k: int(v) for k, v in csv_counts.items()}})
count_table["status"] = (count_table["expected"] == count_table["csv"]).map({True: "PASS", False: "FAIL"})
print(count_table.to_string())

# 2) Values, rates and averages
avg_late_sql = float(pd.read_sql(text(
    "SELECT ROUND(AVG(delivery_delay_days), 2) AS v FROM analytics.orders WHERE in_delay_scope AND is_late"
), engine).iloc[0, 0])
late_items = items_csv[i_scope & items_csv["is_late"].fillna(False)]
price_csv = float(items_csv.loc[i_scope, "price"].sum())
freight_csv = float(items_csv.loc[i_scope, "freight_value"].sum())

value_rows = [
    ("price_in_scope", EXPECTED_VALUES["price_in_scope"], price_csv,
     abs(price_csv - EXPECTED_VALUES["price_in_scope"]) <= TOLERANCE_BRL),
    ("freight_in_scope", EXPECTED_VALUES["freight_in_scope"], freight_csv,
     abs(freight_csv - EXPECTED_VALUES["freight_in_scope"]) <= TOLERANCE_BRL),
    ("freight_share_pct", 14.26, round(freight_csv / (price_csv + freight_csv) * 100, 2), None),
    ("late_rate_pct", 6.77, round(o_late.sum() / o_scope.sum() * 100, 2), None),
    ("avg_score_late", 2.27, round(float(orders_csv.loc[o_late, "review_score"].mean()), 2), None),
    ("avg_score_not_late", 4.29, round(float(orders_csv.loc[o_not_late, "review_score"].mean()), 2), None),
    ("avg_days_late_orders_csv", avg_late_sql, round(float(orders_csv.loc[o_late, "delivery_delay_days"].mean()), 2), None),
    ("avg_days_late_items_csv_per_order", avg_late_sql,
     round(float(late_items.drop_duplicates("order_id")["delivery_delay_days"].mean()), 2), None),
]
value_table = pd.DataFrame(value_rows, columns=["check", "expected", "csv", "ok"]).set_index("check")
value_table["ok"] = value_table["ok"].fillna(value_table["expected"] == value_table["csv"])
value_table["status"] = value_table["ok"].map({True: "PASS", False: "FAIL"})
print("\n" + value_table.drop(columns="ok").to_string())

item_weighted = round(float(late_items["delivery_delay_days"].mean()), 2)
print(f"\nFor comparison only, item-weighted average days late (wrong grain): {item_weighted}")

# 3) D13 identity and order-level columns copied to items
identity_ok = bool((
    orders_csv.loc[o_scope, "actual_delivery_days"] - orders_csv.loc[o_scope, "promised_delivery_days"]
    == orders_csv.loc[o_scope, "delivery_delay_days"]
).all())

copied_cols = ["in_delay_scope", "is_late", "delivery_delay_days", "customer_state"]
paired = items_csv[["order_id"] + copied_cols].merge(
    orders_csv[["order_id"] + copied_cols], on="order_id", how="left",
    validate="many_to_one", suffixes=("_item", "_order"),
)

def same_values(a, b):
    """Equal element by element, treating two missing values as equal."""
    return bool(((a == b) | (a.isna() & b.isna())).fillna(False).all())

copied_ok = {c: same_values(paired[f"{c}_item"], paired[f"{c}_order"]) for c in copied_cols}
print(f"\nD13 identity in scope: {identity_ok}")
print(f"Copied columns equal to orders (hence constant per order): {copied_ok}")

assert (count_table["status"] == "PASS").all()
assert (value_table["status"] == "PASS").all()
assert identity_ok and all(copied_ok.values())

                             expected     csv status
orders                          99441   99441   PASS
delivered                       96478   96478   PASS
delay_scope                     96470   96470   PASS
late                             6534    6534   PASS
delivered_on_estimated_date      1292    1292   PASS
inconsistent_timestamps          1373    1373   PASS
item_rows                      112650  112650   PASS
items_in_scope                 110189  110189   PASS
unknown_item_rows_delivered      1537    1537   PASS
unknown_orders_delivered         1392    1392   PASS
orders_without_review             768     768   PASS
reviewed_in_scope               95824   95824   PASS
reviewed_late                    6381    6381   PASS
reviewed_not_late               89443   89443   PASS
missing_distance_in_scope         481     481   PASS

                                      expected          csv status
check                                                             
price_in_scope   

## Summary

- All control totals reconcile across the reference, the SQL views and pandas, and again from the exported files: 99,441 orders, 96,470 in the delay scope, 6,534 late (6.77%), 110,189 items in scope, price R$ 13,220,248.93, freight R$ 2,198,145.90 (14.26% of order value), 95,824 reviewed orders (average score 4.29 when not late versus 2.27 when late).
- The promised delivery time is about twice the actual one (medians of 24 and 10 days). Extreme values are kept, and the EDA reports medians next to means.
- 95,989 orders in scope have a distance (median 436 km); 481 (0.50%) do not.
- Rules for Excel and Power BI: count late orders from `order_items` with a distinct count of `order_id`, and average delays per order (10.62 days among late orders, against 10.49 if weighted by item).
- Next: `notebooks/03_eda.ipynb`.